In [18]:
from pathlib import Path
from nilearn.glm.first_level import FirstLevelModel
from nilearn.image import mean_img
from nilearn.glm.contrasts import compute_fixed_effects
from nilearn.glm.first_level import FirstLevelModel
from nilearn import image

import nibabel as nib
import numpy as np


In [19]:
inputdir = ""
output_dir = Path.cwd() / "results" / "plot_two_runs_model"
output_dir.mkdir(exist_ok=True, parents=True)
print(f"Output will be saved to: {output_dir}")

Output will be saved to: /Users/clairenastaskin/Documents/Anise/Experiments/results/plot_two_runs_model


In [39]:
# Load and process multiple runs
base_path = "/Users/clairenastaskin/data/2025-03-13/combine_fusi_runs/22mm_34mm/"
runs = sorted(Path(base_path).glob("run*.nii.gz"))
design_mats = sorted(Path(base_path).glob("design_matrix_*.npz"))

# Initialize empty lists to store nifti objects and their data
fusi_imgs = []
func_data = []


# Load each run and store its data
for run in runs:
    nifti = nib.load(run)
    fusi_imgs.append(nifti)
    func_data.append(nifti.get_fdata())
    print(f'Dimensions of {run.stem}: {nifti.shape}')

# Create mean image from first run for plotting
mean_img_ = mean_img(fusi_imgs[0])

# Load design matrices into list of DataFrames
design_matrices = []
for design_mat in design_mats:
    design_matrices.append(np.load(design_mat))
    print(f'Loaded design matrix from {design_mat.stem}')



Dimensions of run1.nii: (85, 39, 42, 235)
Dimensions of run2.nii: (85, 39, 42, 156)
Loaded design matrix from design_matrix_1
Loaded design matrix from design_matrix_2


In [42]:
print(design_matrices[0])

In [38]:

# Initialize and fit the GLM model with specified parameters
print("Fitting a GLM")
fmri_glm = FirstLevelModel(
    minimize_memory=False,
    mask_img=False,
    smoothing_fwhm=smoothing_fwhm,
    standardize=True,
)
fmri_glm = fmri_glm.fit(nifti_data, design_matrices=design_matrix)

In [ ]:
print("Computing contrasts for run 1")
affine_zmap = np.diag([0.15, 0.15, 0.15, 0])

# Iterate on contrasts
for contrast_id, contrast_val in basic_contrasts.items():
    print(f"\tcontrast id: {contrast_id}")
    # compute the contrasts
    z_map = fmri_glm.compute_contrast(contrast_val, output_type="stat")

    # Get the data from z_map and create new NIfTI image
    z_map_data = z_map.get_fdata()
    nii_zmap = nib.Nifti1Image(z_map_data, affine_zmap)

    # Save the NIfTI zmap to a compressed .nii.gz file in the base_path
    output_path = os.path.join(base_path, f"{contrast_id}_z_map.nii.gz")
    nib.save(nii_zmap, output_path)

    # plot the contrasts as soon as they're generated
    # the display is overlaid on the mean fusi image
    # a threshold of 2.58 is used (corresponding to p < 0.01)
    plotting.plot_stat_map(
        z_map,
        bg_img=mean_image,
        threshold=2.58,
        cut_coords=[0],
        display_mode="y",
        black_bg=True,
        title=contrast_id,
    )
    plotting.show()